# Phase 1 — Data Exploration & Cleaning

This notebook loads the raw 7K Books dataset from Kaggle, cleans it, and prepares it for downstream ML tasks.

**Output:** `data/books_cleaned.csv`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

## 1. Load the dataset

In [ ]:
# Download from: https://www.kaggle.com/datasets/dylanjcastillo/7k-books-with-metadata
# Place the CSV at data/books.csv
df = pd.read_csv('../data/books.csv')
print(f'Raw dataset: {df.shape[0]} rows, {df.shape[1]} columns')
df.head()

In [ ]:
df.info()

In [ ]:
# Check missing values per column
df.isnull().sum().sort_values(ascending=False)

## 2. Drop rows with missing critical fields

In [ ]:
required_columns = ['description', 'page_count', 'average_rating', 'published_year']
df.dropna(subset=required_columns, inplace=True)
print(f'After dropping missing critical fields: {len(df)} rows')

## 3. Filter out books with very short descriptions

In [ ]:
# Descriptions shorter than 25 words carry almost no semantic signal
df['word_count'] = df['description'].str.split().str.len()

# Distribution of description lengths
fig, ax = plt.subplots(figsize=(10, 4))
df['word_count'].hist(bins=50, ax=ax)
ax.axvline(25, color='red', linestyle='--', label='25-word threshold')
ax.set_xlabel('Word count')
ax.set_ylabel('Number of books')
ax.set_title('Distribution of description lengths')
ax.legend()
plt.tight_layout()
plt.show()

df = df[df['word_count'] >= 25].copy()
df.drop(columns=['word_count'], inplace=True)
print(f'After filtering short descriptions: {len(df)} rows')

## 4. Merge title and subtitle

In [ ]:
# Format: "Title: Subtitle" only when a non-null subtitle exists
df['title'] = df.apply(
    lambda row: f"{row['title']}: {row['subtitle']}"
    if pd.notna(row.get('subtitle')) and str(row.get('subtitle', '')).strip()
    else row['title'],
    axis=1
)
df[['title', 'subtitle']].sample(5)

## 5. Create `tag_description` (the ISBN hack)

We prepend the `isbn13` to each description. When we later get raw text back from the vector database, we can parse the ISBN from the very first token — without needing any separate metadata store.

In [ ]:
df['tag_description'] = df['isbn13'].astype(str) + ' ' + df['description']
print('Example tag_description:')
print(df['tag_description'].iloc[0][:120], '...')

## 6. Explore the `categories` column (motivation for Phase 2)

In [ ]:
# The raw categories column has hundreds of inconsistent tags
print(f'Unique categories: {df["categories"].nunique()}')
df['categories'].value_counts().head(20)

In [ ]:
# Save the cleaned data
df.reset_index(drop=True, inplace=True)
df.to_csv('../data/books_cleaned.csv', index=False)
print(f'Saved {len(df)} cleaned books to data/books_cleaned.csv')